# RetinaNet — Focal Loss for Dense Object Detection (2017)

_Papers · Computer Vision_

**Reshape cross entropy so easy, well-classified examples are down-weighted, letting a simple one-stage detector match two-stage accuracy.**

---

This notebook is a practice scaffold for the **RetinaNet — Focal Loss for Dense Object Detection (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Define focal loss from scratch

Focal loss (Eqn 5 of the paper) reshapes cross entropy: it multiplies the usual `-log(p_t)` by a modulating factor `(1 - p_t)**gamma` that shrinks the loss for well-classified examples. The `alpha_t` term is a fixed per-class weight that balances positives against negatives. We compute `p_t` — the probability assigned to the *true* class — then assemble the loss.

To prove the implementation is correct, we sanity-check that with `gamma=0` and `alpha=0.5` focal loss collapses back to plain cross entropy (the constant 0.5 factor is undone by multiplying by 2).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Focal loss from scratch (Eqn 5 of arXiv:1708.02002):
#   FL(p_t) = -alpha_t (1-p_t)^gamma log(p_t)
def focal_loss(logits, targets, gamma=2.0, alpha=0.25, reduction="mean"):
    p = torch.sigmoid(logits)                        # predicted P(y=1)
    pt = torch.where(targets == 1, p, 1 - p)         # prob of the TRUE class
    at = torch.where(targets == 1,
                     torch.full_like(p, alpha),
                     torch.full_like(p, 1 - alpha))  # per-class weight
    loss = -at * (1 - pt) ** gamma * torch.log(pt.clamp(min=1e-8))
    return loss.mean() if reduction == "mean" else loss

# Sanity check: gamma=0, alpha=0.5 must equal cross entropy.
# (alpha=0.5 contributes a constant 0.5 factor, so multiply by 2 to undo it.)
torch.manual_seed(0)
logits = torch.randn(64)
targets = (torch.rand(64) > 0.5).float()
fl0 = focal_loss(logits, targets, gamma=0.0, alpha=0.5) * 2
ce0 = F.binary_cross_entropy_with_logits(logits, targets)
print("focal(gamma=0) == cross entropy:", torch.allclose(fl0, ce0, atol=1e-6))
# -> True  (our own check, the loss's built-in oracle)

### Step 2 — Watch easy examples get down-weighted

This is the heart of the paper. Imagine a batch with 1000 easy, well-classified negatives (`p_t = 0.95`) and only 10 hard, uncertain examples (`p_t = 0.50`). Under plain cross entropy the sheer number of easy examples dominates the total loss. Focal loss applies the `(1 - p_t)**gamma` factor, which crushes the easy examples far harder than the hard ones — flipping which group drives the gradient.

In [ ]:
# Easy-example down-weighting, numerically.
# 1000 easy negatives (model 95% sure) + 10 hard (50% sure).
easy_pt = torch.full((1000,), 0.95)   # well-classified -> p_t high
hard_pt = torch.full((10,), 0.50)     # uncertain

def ce(pt):
    return -torch.log(pt)

def fl(pt, g=2.0):
    return -((1 - pt) ** g) * torch.log(pt)

for name, fn in [("CE     ", ce), ("FL g=2 ", lambda pt: fl(pt, 2.0))]:
    e = fn(easy_pt).sum().item()
    h = fn(hard_pt).sum().item()
    easy_share = e / (e + h)
    print(f"{name}: easy_total={e:7.3f}  hard_total={h:6.3f}  easy_share={easy_share:.3f}")
# CE     : easy_total= 51.293  hard_total= 6.931  easy_share=0.881
# FL g=2 : easy_total=  0.128  hard_total= 1.733  easy_share=0.069
# -> CE: easy negatives are 88% of the loss; focal: only 7%. The flip is the whole idea.

### Step 3 — Reproduce the lesson's worked numbers

The lesson quotes exact ratios of cross entropy to focal loss at specific `p_t` values. Since the `-log(p_t)` term cancels in the ratio, `CE / FL` is simply `1 / (1 - p_t)**2` at `gamma=2`. The more confident the model already is, the larger the down-weighting — 100x at `p_t=0.9`, nearly 1000x at `p_t=0.968`, but only 4x at `p_t=0.5`.

In [ ]:
# Worked example from the lesson (must match the text).
for pt in [0.9, 0.968, 0.5]:
    c = -math.log(pt)
    f = -((1 - pt) ** 2) * math.log(pt)
    ratio = c / f
    print(f"p_t={pt:<5}: CE={c:.4f}  FL(g2)={f:.6f}  CE/FL={ratio:7.1f}x")
# p_t=0.9  : CE=0.1054  FL(g2)=0.001054  CE/FL=  100.0x
# p_t=0.968: CE=0.0325  FL(g2)=0.000033  CE/FL=  976.6x
# p_t=0.5  : CE=0.6931  FL(g2)=0.173287  CE/FL=    4.0x

### Step 4 — Train an imbalanced classifier and ablate gamma

Now we put focal loss to work on a 1:99 imbalanced, overlapping toy dataset, where cross entropy is tempted to ignore the rare positive class entirely. We train a tiny linear classifier with each loss and measure **recall on the rare class** — focal loss recovers more positives. Then we sweep `gamma` to reproduce the SHAPE of the paper's Table 1b: recall peaks around `gamma=2` and falls off at both extremes (too low = no focusing, too high = even useful hard examples are starved of gradient).

In [ ]:
# Train a tiny imbalanced classifier: cross entropy vs focal.
# 1:99 imbalance, overlapping classes -> CE is tempted to ignore positives.
N_pos, N_neg = 40, 3960
g = torch.Generator().manual_seed(3)
Xp = torch.randn(N_pos, 2, generator=g) * 1.3 + torch.tensor([1.1, 1.1])
Xn = torch.randn(N_neg, 2, generator=g) * 1.3 + torch.tensor([-0.9, -0.9])
X = torch.cat([Xp, Xn])
y = torch.cat([torch.ones(N_pos), torch.zeros(N_neg)])

def train(kind, gamma=2.0, alpha=0.5, steps=400):
    torch.manual_seed(7)
    w = torch.zeros(2, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    opt = torch.optim.Adam([w, b], lr=0.05)
    for _ in range(steps):
        opt.zero_grad()
        logits = X @ w + b
        if kind == "ce":
            loss = F.binary_cross_entropy_with_logits(logits, y)
        else:
            loss = focal_loss(logits, y, gamma=gamma, alpha=alpha)
        loss.backward()
        opt.step()
    with torch.no_grad():
        pred = (torch.sigmoid(X @ w + b) > 0.5).float()
        tp = ((pred == 1) & (y == 1)).sum().item()
        fn = ((pred == 0) & (y == 1)).sum().item()
        return tp / (tp + fn)                         # recall on the RARE class

print("recall(rare class)  CE   :", round(train("ce"), 3))    # -> 0.1
print("recall(rare class)  focal:", round(train("focal"), 3)) # -> 0.175

# Ablation over gamma (alpha fixed at 0.5 to isolate the focusing effect).
# Reproduces the SHAPE of the paper's Table 1b: peak then fall.
for gm in [0.0, 0.5, 1.0, 2.0, 5.0]:
    print(f"gamma={gm}: recall={round(train('focal', gamma=gm), 3)}")
# gamma=0.0: recall=0.1    (== CE, factor is 1 for everyone)
# gamma=0.5: recall=0.1
# gamma=1.0: recall=0.125
# gamma=2.0: recall=0.175  (best, matches the paper's gamma=2 peak)
# gamma=5.0: recall=0.175  (precision falls -- too aggressive, like the paper's AP drop)
# All numbers above are our small-scale run, not the paper's reported numbers.

## Visualize the data & results

_Under cross entropy, do the many easy negatives dominate the total loss — and does focal loss (gamma=2) flip that, refocusing onto the hard examples?_

### Step 1 — Set up the easy vs hard loss comparison

To visualise where the loss mass goes, we recreate the same scenario as before: 1000 easy negatives (`p_t = 0.95`) and 10 hard examples (`p_t = 0.50`). We define cross entropy and focal loss (`gamma=2`) as plain functions of `p_t` so we can apply each to both groups.

In [ ]:
import torch

# Reproduces focal loss's core effect: where does the loss mass go?
# 1000 easy negatives (p_t=0.95) + 10 hard examples (p_t=0.50).
easy_pt = torch.full((1000,), 0.95)
hard_pt = torch.full((10,), 0.50)

def ce(pt):
    return -torch.log(pt)                            # cross entropy

def fl(pt, g=2.0):
    return -((1 - pt) ** g) * torch.log(pt)          # focal loss (alpha omitted)

### Step 2 — Measure each group's share of the total loss

For each loss we sum over the easy group and the hard group, then report what fraction of the total each contributes. Under cross entropy the easy negatives own ~88% of the loss; under focal loss their share collapses to ~7% and the hard examples take over. That flip is exactly what lets a one-stage detector train despite extreme foreground/background imbalance.

In [ ]:
rows = {}
for name, fn in [("cross entropy", ce), ("focal (g=2)", lambda pt: fl(pt, 2.0))]:
    e = fn(easy_pt).sum().item()
    h = fn(hard_pt).sum().item()
    rows[name] = (e / (e + h), h / (e + h))          # (easy share, hard share)

for name, (es, hs) in rows.items():
    print(f"{name:14s}  easy_share={es:.3f}  hard_share={hs:.3f}")
# cross entropy   easy_share=0.881  hard_share=0.119
# focal (g=2)     easy_share=0.069  hard_share=0.931
# Our small run, not the paper's reported numbers.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** For an easy example with $p_t=0.9$ and $\gamma=2$, by what factor is the focal loss smaller than cross entropy (ignore $\alpha$)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the ratio CE / FL $= \frac{-\log p_t}{-(1-p_t)^2\log p_t}$. — _The $-\log p_t$ cancels, so the ratio is just $1/(1-p_t)^2$ — it depends only on the modulating factor._
- Plug in: $1/(1-0.9)^2 = 1/(0.1)^2 = 1/0.01$. — _$1-p_t = 0.1$; squaring gives $0.01$._

**Answer:** $100\times$ — exactly the figure the paper quotes for $p_t=0.9$ at $\gamma=2$.

</details>

**Problem 2.** A batch is 99% easy negatives ($p_t\approx 0.95$) and 1% hard examples ($p_t\approx 0.5$). Qualitatively, where does most of the cross-entropy total go, and how does $\gamma=2$ change that?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Cross entropy per easy negative is $-\log(0.95)\approx 0.051$; per hard example $-\log(0.5)\approx 0.69$. Multiply each by its count. — _There are ~99× more easy negatives, so even at 0.051 each their TOTAL can exceed the hard examples'._
- Now apply $(1-p_t)^2$: easy negatives get $\times(0.05)^2=\times0.0025$; hard examples get $\times(0.5)^2=\times0.25$. — _Easy mass is crushed ~400× harder than hard mass, flipping which group dominates._

**Answer:** Under cross entropy the easy negatives dominate the total (the notebook below measures ~88% of the loss mass). Under focal loss with $\gamma=2$ their share collapses to ~7%, and the hard examples now drive the gradient.

</details>

**Problem 3.** ABLATION. RetinaNet is best at $\gamma=2$ and worse at both $\gamma=0$ and $\gamma=5$ (Table 1b). Explain both ends.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Consider $\gamma=0$. — _The modulating factor is $(1-p_t)^0=1$ for everyone — focal loss IS cross entropy, so the easy negatives still flood the loss; this is the imbalance problem the paper set out to fix._
- Consider $\gamma=5$. — _The factor decays so steeply that even moderately-hard examples (say $p_t=0.7$) are down-weighted by $(0.3)^5\approx0.0024$, starving the model of useful gradient; too few examples drive learning._

**Answer:** $\gamma$ is a focus dial: too low (0) and easy examples are not suppressed (no better than cross entropy); too high (5) and even useful hard-ish examples are suppressed. $\gamma=2$ balances the two — and our toy run reproduces the same peak-then-fall shape.

</details>